# 2 · The provider's catalog, as the consumer sees it

The consumer asks for the provider's catalog over DSP, with its GNAP token, and picks a dataset to negotiate. These notebooks work with whatever the provider publishes: data of your own, an extension's, or the demo's.

Which dataset: `EUNOMIA_DATASET` (part of its title or id; `./eunomia notebook <provider> <consumer> --dataset "..."` sets it), or else the first one with a policy. To publish something first, use an extension (`./eunomia demo publish <provider>`) or the provider's console.

In [ ]:
%run ./01_auth.ipynb

## The catalog request (consumer → provider)

In [ ]:
payload = {"associatedAgentPeer": provider_did, "filter": [], "noCache": True}
response = requests.post(consumer + "/dsp/current/catalog/rpc/setup-catalog-request", headers=json_header, data=json.dumps(payload))
remote_catalog = show(response, f"{consumer_name} requests {provider_name}'s catalog", open_response=False)["response"]
datasets = remote_catalog.get("dataset") or []
if not datasets:
    raise RuntimeError(f"{provider_name} publishes nothing yet. Publish a dataset first, e.g. ./eunomia demo publish {provider_name}")


def formats(d):
    return [x.get("format") or x.get("formats") or "?" for x in d.get("distribution") or []]


table(
    datasets,
    [
        ("Dataset", "@id"),
        ("Title", "title"),
        ("Formats", lambda d: ", ".join(formats(d))),
        ("Policies", lambda d: len(d.get("hasPolicy") or [])),
    ],
    title=f"{provider_name}'s catalog, as {consumer_name} sees it",
);

## The dataset to negotiate

In [ ]:
wanted = (os.environ.get("EUNOMIA_DATASET") or "").casefold()
usable = [d for d in datasets if d.get("hasPolicy")]
if not usable:
    raise RuntimeError("No dataset has a policy, so none can be negotiated.")
matches = [d for d in usable if wanted in (d.get("title") or "").casefold() or wanted in d["@id"].casefold()]
if not matches:
    raise RuntimeError(f"No dataset matches {wanted!r}. Titles: {[d.get('title') for d in usable]}")
dataset = matches[0]
policy = dataset["hasPolicy"][0]
target_id = dataset["@id"]
policy_id = policy["@id"]
provider_dsp = remote_catalog["service"]["endpointURL"]


def rules(policy, kind):
    out = []
    for rule in policy.get(kind) or []:
        constraints = " and ".join(f"{c['leftOperand']} {c['operator']} {c['rightOperand']}" for c in rule.get("constraint") or [])
        out.append(rule.get("action", "") + (f" if {constraints}" if constraints else ""))
    return "; ".join(out) or "—"


table(
    dataset["hasPolicy"],
    [
        ("Policy", "@id"),
        ("Permission", lambda p: rules(p, "permission")),
        ("Obligation", lambda p: rules(p, "obligation")),
        ("Prohibition", lambda p: rules(p, "prohibition")),
        ("Chosen", lambda p: "◀" if p["@id"] == policy_id else ""),
    ],
    title=f"The offers on {dataset.get('title') or target_id}",
)
table(
    [
        {"what": "Dataset", "value": target_id},
        {"what": "Formats", "value": ", ".join(formats(dataset))},
        {"what": "Policy", "value": policy_id},
        {"what": "Provider DSP endpoint", "value": provider_dsp},
    ],
    [("", "what"), ("", "value")],
);

## The consumer's callback address

Every DSP message names the address where the other side answers. The path depends on the DSP version; the host is the consumer as the provider reaches it from its container.

In [ ]:
response = requests.get(consumer + "/.well-known/dspace-version/2025-1", headers=json_header)
consumer_dsp = consumer_docker + show(response, "DSP 2025-1 on the consumer")["path"]
print(f"Consumer callback  {consumer_dsp}")